In [ ]:
!pip install catboost

In [ ]:
#Ignore it this is from me

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder, MinMaxScaler
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import f1_score, accuracy_score
from catboost import CatBoostClassifier

In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
full_path = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(full_path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
df

In [ ]:
# Task 1: Write your code here:
# Let us see the proportion of each feature
# Analyze missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
# And this is the total number of each one
df.isnull().sum()

In [ ]:
df.dtypes.unique() # since they are all numerical values (no categoricals) so we will handle it in a numerical way

In [ ]:
# it is time to work on each numerical feature # for me I prefer to give it each missing with the mean
df_clean = df.copy()
for col in df.columns:
    df_clean[col] = df_clean[col].fillna(df[col].mean())

df_clean.isnull().sum()

In [ ]:
# Analyze missing values for clean
missing_percentage = (df_clean.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data

In [ ]:
# Task 2: Write your code here:
df_clean.duplicated().sum() # there are no dubs

In [ ]:
# Task 3: Write your code here:
# There are no categorical variables, and as we can see below the traget label already encoded
df_clean["Target"].value_counts()

In [ ]:
# Task 4: Write your code here:

X = df_clean.drop(["Target"], axis=1) # we must not include the target
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
df_clean.shape

In [ ]:
X_scaled.shape # just removed the target label and scaled it

In [ ]:
# Task 5: Write your code here:
# Condition distribution


plt.figure(figsize=(10, 5))
plt.bar(df_clean['Target'].unique(),df_clean['Target'].value_counts(), color='green')
plt.title('Target Distribution')
plt.xlabel('Condition')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.grid()
plt.show()

# As we can see here it is clearly that the target is imbalanced (most of them are No Default)

In [ ]:
# Task 1: Write your code here:
# X_scaled already did it before in part two
y = df_clean["Target"].to_numpy()

In [ ]:
X_scaled.shape, y.shape

In [ ]:
# Task 2,3,4,5: Write your code here:

# Define Model
model = CatBoostClassifier()

# K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
f1 = []
accuracy = []
i = 0
for train_idx, test_idx in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Evaluation metrics
    accuracy.append(accuracy_score(y_test, y_pred))
    f1.append(f1_score(y_test, y_pred))

    # Print Evaluation Metrics
    print(f"K-Fold {i + 1}")
    print(f"Accuracy {accuracy[i]}")
    print(f"F1 {f1[i]}")
    print("-"*40)
    i+=1


# Print Evaluation Metrics
print("\nModel Evaluation Metrics (K-Fold)\n" + "-"*40)
print(f"Accuracy Avg : {np.mean(accuracy):.2f}")
print(f"F1 Avg : {np.mean(f1):.2f}")
print("-"*40)

In [ ]:
df_clean.columns

In [ ]:
# Task 1: Write your code here:
# must have those:
feature_cols = df_clean.drop(["Target"], axis=1).columns

# + model (sklearn)

feature_importance = pd.DataFrame({
    'feature': feature_cols[:10],
    'importance': model.feature_importances_[:10]
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()




In [ ]:
# Task 2: Write your code here:

# due to the large number of features I printed the first 10, and finally I can say:
# I FOUND THE GOLDEN FEATURE !!

# it is: "P_2"

In [ ]:
# Task Bonus: Write your code here:
X_bonus = df_clean["P_2"].to_numpy()
scaler = StandardScaler()
X_scaled_bonus = scaler.fit_transform(X_bonus.reshape(-1,1))

In [ ]:
X_scaled_bonus.shape, y.shape

In [ ]:
# Task 2,3,4,5: Write your code here:

# Define Model
model = CatBoostClassifier()

# K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
f1 = []
accuracy = []
i = 0
for train_idx, test_idx in kf.split(X_scaled_bonus):
    X_train, X_test = X_scaled_bonus[train_idx], X_scaled_bonus[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Evaluation metrics
    accuracy.append(accuracy_score(y_test, y_pred))
    f1.append(f1_score(y_test, y_pred))

    # Print Evaluation Metrics
    print(f"K-Fold {i + 1}")
    print(f"Accuracy {accuracy[i]}")
    print(f"F1 {f1[i]}")
    print("-"*40)
    i+=1


# Print Evaluation Metrics
print("\nModel Evaluation Metrics (K-Fold)\n" + "-"*40)
print(f"Accuracy Avg : {np.mean(accuracy):.2f}")
print(f"F1 Avg : {np.mean(f1):.2f}")
print("-"*40)

In [ ]:
# For golden feature, this is the result:

# Model Evaluation Metrics (K-Fold)
# ----------------------------------------
# Accuracy Avg : 0.79
# F1 Avg : 0.58
# ----------------------------------------


# For the whole features, this is the result


# Model Evaluation Metrics (K-Fold)
# ----------------------------------------
# Accuracy Avg : 0.84
# F1 Avg : 0.70
# ----------------------------------------



# "More features are better than a single one"
# of course, but as we can see here the result for golden feature is not that bad since it is single and achive that, I think it is great ;)